# OpenCode Zen / Go gateway (Phase GW) — offline

> Notebook generated from [`examples/gateways_opencode.py`](https://github.com/prismal-ai/prismal/blob/main/examples/gateways_opencode.py).

| Field | Value |
|------|-------|
| **API key** | 🔑 Not required — nothing leaves the process |
| **Source** | `examples/gateways_opencode.py` |
| **Guide** | [`docs/gateways.md`](https://github.com/prismal-ai/prismal/blob/main/docs/gateways.md) |

OpenCode exposes two gateways with different economics and rules:

* **Zen** — pay-per-use, but each model family speaks its own **wire protocol** (Anthropic Messages, OpenAI Responses, Chat Completions).
* **Go** — a flat monthly fee, usable **only by coding agents** (ToS), with rolling quota windows.

The notebook shows:

1. Per-model wire protocols resolved by the same router.
2. The OpenCode Go ToS guard — non-coding agents are refused before any request.
3. The Go flat-fee `QuotaLedger`: an exhausted 5 h window downgrades (or refuses) the call before any network call.

In [1]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · build the router from settings

`build_gateway_router()` composes every enabled gateway for one tenant (`org_id`) and attaches the quota ledger.

In [2]:
from __future__ import annotations

from pydantic import SecretStr

from prismal.core.config import Settings
from prismal.core.exceptions import GatewayPolicyError
from prismal.providers.gateways.router import build_gateway_router

settings = Settings(
    llm_gateways_enabled=True,
    opencode_zen_enabled=True,
    opencode_go_enabled=True,
    opencode_go_monthly_cap_usd=10.0,
    opencode_go_quota_action="downgrade",
    opencode_go_downgrade_model="gw:opencode-go/glm-5.3-flash",
)
router = build_gateway_router(
    settings, org_id="acme", secret_resolver=lambda ref: SecretStr("oc-demo-key")
)
assert router is not None

## 1 · Per-model wire protocols (Zen)

In [3]:
for ref in ("gw:opencode/claude-sonnet-4-5", "gw:opencode/gpt-5.5", "gw:opencode/glm-5.3"):
    resolved = router.resolve(ref, agent="researcher", session_id="sess-1")
    assert resolved is not None
    print(f"{ref:32} -> {resolved.litellm_model:34} [{resolved.wire_protocol}]")

gw:opencode/claude-sonnet-4-5    -> anthropic/claude-sonnet-4-5        [anthropic_messages]
gw:opencode/gpt-5.5              -> openai/responses/gpt-5.5           [openai_responses]
gw:opencode/glm-5.3              -> openai/glm-5.3                     [openai_chat]


## 2 · OpenCode Go ToS guard

The `coder` agent may use Go (`cost_mode=subscription` ⇒ metered `cost_usd=0` plus an `equivalent_usd`); `researcher` is refused.

In [4]:
resolved = router.resolve("gw:opencode-go/glm-5.3", agent="coder", session_id="sess-1")
assert resolved is not None
print("coder      ->", resolved.litellm_model, "| cost_mode =", resolved.cost_mode)
print("headers    :", sorted(resolved.extra_headers))
try:
    router.resolve("gw:opencode-go/glm-5.3", agent="researcher", session_id="sess-1")
except GatewayPolicyError as exc:
    print("researcher -> refused:", exc.reason)

coder      -> openai/glm-5.3 | cost_mode = subscription
headers    : ['User-Agent', 'x-opencode-session']


[info     ] gateway.policy_denied          agent=researcher gateway=opencode-go reason=opencode_go_agent_not_allowed


researcher -> refused: opencode_go_agent_not_allowed


## 3 · Go quota ledger

A $10/month cap gives a $2 budget per 5 h window. After recording $2 of equivalent usage, the preflight verdict for the next call is a downgrade to the configured cheaper model.

In [5]:
ledger = router.ledger
assert ledger is not None
ledger.record("opencode-go", "glm-5.3", equivalent_usd=2.0)
verdict = ledger.preflight("opencode-go", "glm-5.3", est_equivalent_usd=0.01)
print(f"verdict: {verdict.action} (window {verdict.exhausted_window})")
print("downgrade target:", resolved.downgrade_model)
for snap in ledger.snapshot("opencode-go"):
    print(f"{snap.window:>5}: used ${snap.used:.2f} of ${snap.limit:.2f}")

verdict: downgrade (window 5h)
downgrade target: openai/glm-5.3-flash
   5h: used $2.00 of $2.00
 week: used $2.00 of $5.00
month: used $2.00 of $10.00


---

After a downgrade, `BudgetGuard.degradation()` reports `gateway_quota:<window>`, so the budget layer and the host read surface see the same event.